# Agent 实现长任务的各种方式

当我们要求 AI Agent 完成一个复杂的长任务时——例如「帮我规划一次旅行，包括机票、酒店、行程和预算」——简单的一次性 Prompt 往往不够。

本 Notebook 将 **从简单到复杂**，逐步探索 Agent 处理长任务的 **7 种核心模式**：

| # | 模式 | 核心思想 | 适用场景 |
|---|------|---------|----------|
| 1 | 顺序链 (Sequential Chain) | 固定流水线，步步推进 | 流程明确的任务 |
| 2 | 任务分解 (Task Decomposition) | 先规划再执行 | 复杂多步骤任务 |
| 3 | ReAct 循环 | 思考 → 行动 → 观察，循环往复 | 需要动态决策的任务 |
| 4 | 工具调用 (Tool Use) | Agent 调用外部函数获取信息 | 需要实时数据的任务 |
| 5 | 多 Agent 协作 | 专业分工，协同完成 | 跨领域复杂任务 |
| 6 | 反思与自我修正 (Reflection) | 检查结果，迭代改进 | 高质量要求的任务 |
| 7 | 人机协作 (Human-in-the-Loop) | 关键节点人工确认 | 高风险决策任务 |

> **运行要求**：本 Notebook **不需要 API Key**。所有 Agent 行为通过模拟函数实现，让你专注于理解模式本身。学会模式后，可以轻松替换为真实的 LLM 调用（Semantic Kernel / AutoGen）。

## 准备工作：模拟 Agent 基础设施

我们先构建一个轻量的模拟框架，让后续每种模式的代码更清晰。

In [ ]:
import time
import json
import random
import textwrap
from dataclasses import dataclass, field
from typing import Any, Callable
from enum import Enum


class Role(Enum):
    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"
    TOOL = "tool"


@dataclass
class Message:
    role: Role
    content: str
    name: str = ""

    def __repr__(self):
        prefix = f"[{self.name}] " if self.name else ""
        tag = self.role.value.upper()
        short = self.content[:120].replace('\n', ' ')
        return f"{prefix}{tag}: {short}{'...' if len(self.content) > 120 else ''}"


def log_step(icon: str, title: str, detail: str = ""):
    """格式化输出一个步骤"""
    print(f"\n{icon} {title}")
    if detail:
        for line in detail.strip().split("\n"):
            print(f"   {line}")


def log_divider(title: str = ""):
    if title:
        print(f"\n{'='*20} {title} {'='*20}")
    else:
        print("─" * 60)


print("✅ 基础设施就绪")

---
## 模式 1：顺序链 (Sequential Chain)

最简单的方式：将长任务拆成 **固定的步骤序列**，每步的输出作为下一步的输入。

```
用户需求 → [步骤1: 理解需求] → [步骤2: 搜索信息] → [步骤3: 生成方案] → [步骤4: 格式化输出] → 最终结果
```

### 优点
- 简单直观，易于调试
- 每一步可以用不同的 Prompt / 模型

### 缺点
- 步骤固定，无法根据中间结果动态调整
- 错误会在链条中传播

### 对应框架
- Semantic Kernel: 多个 `KernelFunction` 顺序调用
- LangChain: `SequentialChain`

In [ ]:
def sequential_chain(user_request: str) -> str:
    """顺序链：固定步骤的流水线"""

    log_divider("模式1: 顺序链")
    log_step("📥", "用户输入", user_request)

    # 步骤1: 理解需求
    log_step("🔍", "步骤1 — 需求分析")
    analysis = {
        "destination": "Tokyo",
        "duration": "5 days",
        "budget": "moderate",
        "interests": ["culture", "food", "technology"]
    }
    print(f"   提取的关键信息: {json.dumps(analysis, indent=2, ensure_ascii=False)}")

    # 步骤2: 搜索信息
    log_step("🌐", "步骤2 — 信息搜索")
    search_results = {
        "flights": "北京→东京 往返 ¥3200 (ANA)",
        "hotels": "新宿华盛顿酒店 ¥600/晚",
        "attractions": ["浅草寺", "秋叶原", "teamLab", "筑地市场", "明治神宫"]
    }
    for k, v in search_results.items():
        print(f"   {k}: {v}")

    # 步骤3: 生成方案
    log_step("📝", "步骤3 — 生成行程")
    itinerary = [
        "Day 1: 抵达东京，新宿入住，晚上歌舞伎町散步",
        "Day 2: 浅草寺 → 晴空塔 → 秋叶原电器街",
        "Day 3: 明治神宫 → 原宿竹下通 → 涩谷十字路口",
        "Day 4: teamLab → 台场 → 自由活动",
        "Day 5: 筑地市场早餐 → 返程"
    ]
    for day in itinerary:
        print(f"   {day}")

    # 步骤4: 汇总输出
    log_step("✅", "步骤4 — 最终方案")
    total = "机票¥3200 + 酒店¥3000(5晚) + 餐饮¥2000 + 门票¥800 = 总计约¥9000"
    print(f"   预算: {total}")

    return f"5日东京之旅方案（预算{total}）"


result = sequential_chain("帮我规划一次5天的东京旅行，我喜欢文化和美食")
print(f"\n🎯 最终结果: {result}")

### 思考

顺序链的问题很明显：如果步骤 2 搜索到的航班太贵，整个流程无法「回头」调整。我们需要更灵活的方式。

---
## 模式 2：任务分解 (Task Decomposition / Planning)

核心思想：**先规划，再执行**。Agent 先将复杂任务分解为子任务列表，然后逐个完成。

```
用户需求
  ↓
[Planner Agent] → 生成任务计划 (Plan)
  ↓
  ├─ 子任务1 → 执行 → 结果1
  ├─ 子任务2 → 执行 → 结果2
  ├─ 子任务3 → 执行 → 结果3
  ↓
[Summarizer] → 汇总所有结果 → 最终输出
```

### 关键改进
- 子任务可以并行 / 按依赖关系排序
- 可以动态增删子任务
- 每个子任务可以交给不同的专家 Agent

### 对应框架
- Semantic Kernel: `HandlebarsPlanner` / `FunctionCallingStepwisePlanner`
- AutoGen: Planner Agent 生成结构化 JSON
- 课程 Lesson 07 详细讲解了这个模式

In [ ]:
@dataclass
class SubTask:
    id: int
    description: str
    agent: str
    depends_on: list[int] = field(default_factory=list)
    status: str = "pending"
    result: str = ""


def plan_task(user_request: str) -> list[SubTask]:
    """Planner Agent: 将用户需求分解为子任务"""
    log_step("🧠", "Planner: 分析任务并制定计划")

    plan = [
        SubTask(1, "分析用户偏好和约束条件", "analyzer"),
        SubTask(2, "搜索符合条件的航班", "flight_agent", depends_on=[1]),
        SubTask(3, "搜索目的地酒店", "hotel_agent", depends_on=[1]),
        SubTask(4, "规划每日行程", "itinerary_agent", depends_on=[1]),
        SubTask(5, "计算总预算", "budget_agent", depends_on=[2, 3, 4]),
        SubTask(6, "生成最终方案报告", "summarizer", depends_on=[2, 3, 4, 5]),
    ]

    print("   生成的执行计划:")
    for t in plan:
        deps = f" (依赖: {t.depends_on})" if t.depends_on else " (无依赖)"
        print(f"   [{t.id}] {t.description} → @{t.agent}{deps}")

    return plan


def execute_subtask(task: SubTask, context: dict) -> str:
    """执行单个子任务"""
    results = {
        1: "偏好: 文化+美食, 预算: 中等, 时间: 5天, 出发: 北京",
        2: "推荐航班: ANA NH964 北京-东京 ¥3200往返 (直飞4h)",
        3: "推荐酒店: 新宿格拉斯丽酒店 ¥550/晚 (评分4.5/5)",
        4: "5日行程: 浅草→秋叶原→明治神宫→teamLab→筑地市场",
        5: "总预算: ¥3200(机票)+¥2750(酒店)+¥2500(餐饮)+¥800(门票)=¥9250",
        6: "完整方案已生成，包含航班、住宿、行程和预算明细",
    }
    return results.get(task.id, "未知任务")


def task_decomposition(user_request: str):
    """任务分解模式：先规划再执行"""

    log_divider("模式2: 任务分解")
    log_step("📥", "用户输入", user_request)

    # Phase 1: 规划
    plan = plan_task(user_request)

    # Phase 2: 按依赖顺序执行
    log_step("⚡", "开始执行子任务")
    completed = set()
    context = {}

    while len(completed) < len(plan):
        for task in plan:
            if task.id in completed:
                continue
            if all(dep in completed for dep in task.depends_on):
                task.status = "running"
                task.result = execute_subtask(task, context)
                task.status = "done"
                context[task.id] = task.result
                completed.add(task.id)
                print(f"   ✅ [{task.id}] {task.description}")
                print(f"      → {task.result}")

    log_step("🎯", "所有子任务完成", f"执行了 {len(plan)} 个子任务")
    return context


task_decomposition("帮我规划一次5天的东京旅行，我喜欢文化和美食，预算中等")

### 关键观察

注意子任务 2、3、4 没有互相依赖 → 它们可以 **并行执行**，大幅提高效率。

而子任务 5（预算）依赖 2、3、4 的结果 → 必须等它们都完成后才能开始。

这就是 **DAG（有向无环图）** 调度的核心思想。

---
## 模式 3：ReAct 循环 (Reasoning + Acting)

ReAct 是目前最流行的 Agent 推理范式。Agent 不再按固定步骤执行，而是在每一步：

1. **Thought (思考)**：分析当前状态，决定下一步做什么
2. **Action (行动)**：执行一个动作（调用工具、搜索信息等）
3. **Observation (观察)**：获取行动的结果
4. **重复**，直到任务完成

```
Thought: 用户要去东京，我需要先查机票价格
Action: search_flights("北京", "东京")
Observation: 找到3个航班，最便宜¥2800
Thought: 机票在预算内，接下来查酒店
Action: search_hotels("东京", "新宿")
Observation: 找到5个酒店，推荐xxx
Thought: 信息足够了，可以生成方案
Action: finish(方案)
```

### 为什么 ReAct 更强大
- Agent 可以根据中间结果 **动态决定下一步**
- 遇到意外情况可以改变策略
- 自然地支持工具调用

### 对应框架
- Semantic Kernel: `FunctionCallingStepwisePlanner`
- AutoGen: `AssistantAgent` with `tools` + `reflect_on_tool_use=True`

In [ ]:
# 定义 Agent 可用的工具
TOOLS = {
    "search_flights": lambda orig, dest: f"找到3个航班: {orig}→{dest} 最低¥2800(春秋), ¥3200(ANA), ¥3800(JAL)",
    "search_hotels": lambda city, area: f"{city}{area}区域5家酒店: 格拉斯丽¥550/晚, 世纪南悦¥480/晚, 希尔顿¥1200/晚",
    "get_attractions": lambda city: f"{city}热门景点: 浅草寺(免费), 秋叶原, teamLab(¥200), 明治神宫(免费), 东京塔(¥100)",
    "calculate_budget": lambda items: f"总预算: {sum(items.values())} 元 (明细: {items})",
    "finish": lambda summary: summary,
}


def react_loop(user_request: str, max_steps: int = 8):
    """ReAct 循环：思考-行动-观察"""

    log_divider("模式3: ReAct 循环")
    log_step("📥", "用户输入", user_request)

    # 模拟 Agent 的 ReAct 推理过程
    trace = [
        ("Thought", "用户想去东京旅行5天，喜欢文化和美食。先搜索航班价格看看预算是否可行。"),
        ("Action", "search_flights('北京', '东京')"),
        ("Observation", TOOLS["search_flights"]("北京", "东京")),

        ("Thought", "最便宜¥2800，中等预算应该选ANA¥3200，直飞舒适。接下来查酒店。"),
        ("Action", "search_hotels('东京', '新宿')"),
        ("Observation", TOOLS["search_hotels"]("东京", "新宿")),

        ("Thought", "格拉斯丽¥550/晚性价比高。5晚=¥2750。再查景点安排行程。"),
        ("Action", "get_attractions('东京')"),
        ("Observation", TOOLS["get_attractions"]("东京")),

        ("Thought", "景点信息齐全了。免费景点较多，门票花费不大。汇总预算。"),
        ("Action", "calculate_budget({'机票': 3200, '酒店': 2750, '餐饮': 2500, '门票': 300, '交通': 500})"),
        ("Observation", TOOLS["calculate_budget"]({"机票": 3200, "酒店": 2750, "餐饮": 2500, "门票": 300, "交通": 500})),

        ("Thought", "总预算¥9250，在中等预算范围内。信息充足，可以生成最终方案。"),
        ("Action", "finish('5天东京之旅方案：ANA直飞+新宿格拉斯丽+文化美食行程，总预算¥9250')"),
    ]

    icons = {"Thought": "💭", "Action": "⚡", "Observation": "👁️"}

    for i, (step_type, content) in enumerate(trace):
        step_num = i // 3 + 1 if step_type == "Thought" else ""
        prefix = f"Step {step_num} " if step_num else "       "
        icon = icons[step_type]
        print(f"\n  {icon} {step_type}: {content}")

        if step_type == "Action" and "finish" in content:
            log_step("🎯", "任务完成", content.split("'")[1])
            break


react_loop("帮我规划一次5天的东京旅行，预算中等，喜欢文化和美食")

### ReAct 的灵活性

注意 Agent 在第 2 步发现最便宜航班¥2800（春秋航空），但 **主动选择了** ¥3200 的 ANA 直飞——因为它推理出「中等预算 + 舒适度」更合适。

这种 **基于中间结果的动态决策** 是顺序链做不到的。

---
## 模式 4：工具调用 (Tool Use / Function Calling)

Agent 的真正威力在于能 **调用外部工具**。工具可以是 API 调用、数据库查询、代码执行等。

在实际框架中：
- **Semantic Kernel**: 用 `@kernel_function` 装饰器注册工具（Plugin）
- **AutoGen**: 用 `FunctionTool` 包装 Python 函数
- **OpenAI API**: `tools` 参数 + `function_calling`

下面我们实现一个完整的工具调用流程，展示 Agent 如何决定调用哪个工具、传什么参数。

In [ ]:
@dataclass
class ToolDef:
    name: str
    description: str
    parameters: dict
    func: Callable


class ToolAgent:
    """支持工具调用的 Agent"""

    def __init__(self, name: str, instructions: str, tools: list[ToolDef]):
        self.name = name
        self.instructions = instructions
        self.tools = {t.name: t for t in tools}
        self.history: list[Message] = []

    def _decide_tool_call(self, user_msg: str) -> list[tuple[str, dict]]:
        """模拟 LLM 决定调用哪些工具（实际中由 LLM 的 function_calling 完成）"""
        calls = []
        msg = user_msg.lower()
        if "天气" in msg or "weather" in msg:
            calls.append(("get_weather", {"city": "东京"}))
        if "航班" in msg or "机票" in msg or "flight" in msg:
            calls.append(("search_flights", {"origin": "北京", "destination": "东京", "date": "2026-04-01"}))
        if "酒店" in msg or "hotel" in msg:
            calls.append(("search_hotels", {"city": "东京", "checkin": "2026-04-01", "nights": 5}))
        if "汇率" in msg or "exchange" in msg:
            calls.append(("get_exchange_rate", {"from_currency": "CNY", "to_currency": "JPY"}))
        if not calls:
            calls.append(("search_flights", {"origin": "北京", "destination": "东京", "date": "2026-04-01"}))
            calls.append(("search_hotels", {"city": "东京", "checkin": "2026-04-01", "nights": 5}))
            calls.append(("get_weather", {"city": "东京"}))
            calls.append(("get_exchange_rate", {"from_currency": "CNY", "to_currency": "JPY"}))
        return calls

    def run(self, user_msg: str):
        """完整的工具调用流程"""
        log_step("👤", f"用户 → {self.name}", user_msg)

        # Step 1: LLM 决定调用哪些工具
        tool_calls = self._decide_tool_call(user_msg)
        log_step("🧠", f"{self.name}: 决定调用 {len(tool_calls)} 个工具")

        # Step 2: 执行工具调用
        tool_results = []
        for tool_name, params in tool_calls:
            if tool_name in self.tools:
                tool = self.tools[tool_name]
                result = tool.func(**params)
                tool_results.append((tool_name, result))
                print(f"   🔧 {tool_name}({params})")
                print(f"      → {result}")

        # Step 3: LLM 综合工具结果生成回复
        log_step("💬", f"{self.name}: 综合工具结果生成回复")
        response = f"根据查询结果：\n"
        for name, result in tool_results:
            response += f"  • {result}\n"
        response += "建议选择 ANA 直飞航班 + 新宿格拉斯丽酒店，4月初东京樱花季正好！"
        print(f"   {response}")

        return response


# 定义工具
travel_tools = [
    ToolDef("search_flights", "搜索航班信息",
            {"origin": "str", "destination": "str", "date": "str"},
            lambda origin, destination, date: f"{origin}→{destination} ({date}): ANA¥3200直飞 | 春秋¥2800转机 | JAL¥3800直飞"),

    ToolDef("search_hotels", "搜索酒店",
            {"city": "str", "checkin": "str", "nights": "int"},
            lambda city, checkin, nights: f"{city}酒店({nights}晚): 格拉斯丽¥{550*nights} | 世纪南悦¥{480*nights} | 希尔顿¥{1200*nights}"),

    ToolDef("get_weather", "查询天气预报",
            {"city": "str"},
            lambda city: f"{city}未来5天: 晴 18°C → 多云 16°C → 晴 20°C → 小雨 14°C → 晴 19°C"),

    ToolDef("get_exchange_rate", "查询汇率",
            {"from_currency": "str", "to_currency": "str"},
            lambda from_currency, to_currency: f"{from_currency}→{to_currency}: 1 CNY = 21.5 JPY"),
]

agent = ToolAgent(
    name="TravelAssistant",
    instructions="你是一个旅行规划助手，可以搜索航班、酒店、天气和汇率信息。",
    tools=travel_tools
)

log_divider("模式4: 工具调用")
agent.run("我想4月份去东京旅行5天，帮我查一下机票、酒店、天气和汇率")

### 工具调用在真实框架中的写法

```python
# === Semantic Kernel ===
class TravelPlugin:
    @kernel_function(description="搜索航班")
    def search_flights(self, origin: str, dest: str) -> str:
        return api.search_flights(origin, dest)

agent = ChatCompletionAgent(
    service=openai_service,
    plugins=[TravelPlugin()],  # 注册工具
)

# === AutoGen ===
search_tool = FunctionTool(
    search_flights,
    description="搜索航班信息"
)
agent = AssistantAgent(
    name="travel",
    model_client=client,
    tools=[search_tool],  # 注册工具
    reflect_on_tool_use=True,
)
```

---
## 模式 5：多 Agent 协作 (Multi-Agent)

当任务涉及多个领域时，可以让 **多个专业 Agent 协作**，每个 Agent 负责自己擅长的部分。

常见的协作方式：

| 方式 | 说明 | 示例 |
|------|------|------|
| 轮询 (Round Robin) | Agent 轮流发言 | 辩论、头脑风暴 |
| 路由 (Router) | 中央调度分配任务 | 客服系统 |
| 分层 (Hierarchical) | 管理者分配，工人执行 | 项目管理 |
| 协商 (Negotiation) | Agent 之间讨论达成共识 | 方案评审 |

### 对应框架
- Semantic Kernel: `AgentGroupChat` + `SelectionStrategy` + `TerminationStrategy`
- AutoGen: `RoundRobinGroupChat` / `GroupChatManager`
- 课程 Lesson 08 详细讲解了这个模式

In [ ]:
@dataclass
class SimpleAgent:
    name: str
    role: str
    expertise: str

    def respond(self, task: str, context: list[str]) -> str:
        """模拟 Agent 回复（实际中调用 LLM）"""
        responses = {
            "FlightExpert": f"航班方案: 推荐ANA NH964直飞，4月樱花季票价¥3200，含23kg托运行李。经济舱座位选靠窗可观赏富士山。",
            "HotelExpert": f"住宿方案: 推荐新宿格拉斯丽酒店(哥斯拉楼)，¥550/晚，地铁直达各景点，楼下就是歌舞伎町。含早餐。",
            "FoodExpert": f"美食攻略: Day1-一蘭拉面(新宿), Day2-寿司大(筑地), Day3-烤肉叙々苑, Day4-天妇罗近藤, Day5-便利店早餐+机场拉面。人均每天¥500。",
            "BudgetExpert": f"预算审核: 机票¥3200+酒店¥2750+餐饮¥2500+门票¥300+交通¥500=¥9250。在中等预算范围内(¥8000-¥12000)。建议预留¥1000应急金。",
            "Coordinator": self._coordinate(context),
        }
        return responses.get(self.name, "无法处理")

    def _coordinate(self, context: list[str]) -> str:
        if not context:
            return "请各位专家依次提供方案。"
        if len(context) >= 4:
            return "所有专家方案已收集完毕。方案整体协调一致，预算合理。最终方案: 5天东京文化美食之旅，总预算¥9250+¥1000应急=¥10250。方案通过！APPROVE"
        return "继续，下一位专家请发言。"


def multi_agent_collaboration(user_request: str):
    """多 Agent 协作模式"""

    log_divider("模式5: 多Agent协作")
    log_step("📥", "用户输入", user_request)

    # 创建专家团队
    agents = [
        SimpleAgent("Coordinator", "协调者", "项目管理与最终决策"),
        SimpleAgent("FlightExpert", "航班专家", "航班搜索与推荐"),
        SimpleAgent("HotelExpert", "酒店专家", "酒店搜索与推荐"),
        SimpleAgent("FoodExpert", "美食专家", "餐厅推荐与美食攻略"),
        SimpleAgent("BudgetExpert", "预算专家", "费用计算与预算审核"),
    ]

    log_step("👥", "Agent 团队", "\n".join(f"@{a.name} — {a.role} ({a.expertise})" for a in agents))

    # 协作对话
    conversation: list[str] = []
    round_num = 0

    # 协调者开场
    log_step("🎬", "协作开始")

    # 各专家轮流发言
    speaking_order = [agents[0], agents[1], agents[2], agents[3], agents[4], agents[0]]

    for agent in speaking_order:
        round_num += 1
        response = agent.respond(user_request, conversation)
        conversation.append(response)

        icon = {"Coordinator": "👔", "FlightExpert": "✈️", "HotelExpert": "🏨",
                "FoodExpert": "🍣", "BudgetExpert": "💰"}.get(agent.name, "🤖")
        print(f"\n  {icon} @{agent.name} ({agent.role}):")
        print(f"     {response}")

        if "APPROVE" in response:
            log_step("✅", "协调者批准方案", "多Agent协作完成!")
            break

    print(f"\n  📊 协作统计: {round_num} 轮对话, {len(agents)} 个Agent参与")


multi_agent_collaboration("帮我规划5天东京旅行，喜欢文化和美食，预算中等")

### 关键要素

多 Agent 协作的三个核心设计决策：

1. **选择策略 (Selection Strategy)**：下一个该谁说话？
   - 轮询、LLM 动态选择、基于规则

2. **终止策略 (Termination Strategy)**：什么时候结束？
   - 关键词（如 "APPROVE"）、最大轮数、LLM 判断

3. **消息路由**：谁能看到谁的消息？
   - 全部可见、仅协调者可见、选择性可见

---
## 模式 6：反思与自我修正 (Reflection & Metacognition)

Agent 完成任务后，增加一个 **反思** 步骤：检查自己的输出，发现问题后自动修正。

```
    ┌─────────────────────────────────┐
    │                                 │
    ▼                                 │ 不通过
  生成方案 → 自我评估 → 通过? ──────→ 修正方案
                         │
                         ▼ 通过
                      输出最终结果
```

### 反思的层次
- **输出检查**：结果是否完整、格式是否正确
- **逻辑检查**：推理过程是否合理
- **策略调整**：下次用更好的方法

### 对应框架
- Semantic Kernel: Writer Agent + Reviewer Agent in `AgentGroupChat`
- AutoGen: `reflect_on_tool_use=True`
- 课程 Lesson 09 详细讲解了元认知

In [ ]:
def generate_plan(attempt: int) -> dict:
    """模拟生成旅行方案（第 attempt 次尝试）"""
    if attempt == 1:
        return {
            "flights": "春秋航空 ¥2800 (转机，耗时9小时)",
            "hotel": "胶囊旅馆 ¥150/晚",
            "itinerary": ["Day1: 到达", "Day2: 逛街", "Day3-5: 自由活动"],
            "budget": "¥5050",
            "issues": ["转机航班太折腾", "住宿条件差", "行程太空没有规划", "没有美食推荐"]
        }
    elif attempt == 2:
        return {
            "flights": "ANA NH964 ¥3200 (直飞，4小时)",
            "hotel": "新宿格拉斯丽 ¥550/晚",
            "itinerary": [
                "Day1: 抵达+新宿夜游+一蘭拉面",
                "Day2: 浅草寺+晴空塔+秋叶原",
                "Day3: 明治神宫+原宿+涩谷",
                "Day4: teamLab+台场+烤肉晚餐",
                "Day5: 筑地市场早餐+返程"
            ],
            "budget": "¥9250",
            "issues": []
        }
    return {}


def evaluate_plan(plan: dict) -> tuple[bool, list[str]]:
    """Reviewer: 评估方案质量"""
    issues = []

    if "转机" in plan.get("flights", ""):
        issues.append("航班有转机，体验差，建议直飞")
    if "胶囊" in plan.get("hotel", "") or "青旅" in plan.get("hotel", ""):
        issues.append("住宿条件低于中等预算标准")
    if len(plan.get("itinerary", [])) < 5:
        issues.append("行程不够详细，应该每天都有具体安排")
    if any("自由活动" in day for day in plan.get("itinerary", [])):
        issues.append("有太多自由活动，缺乏具体推荐")
    if not any("拉面" in d or "寿司" in d or "烤肉" in d for d in plan.get("itinerary", [])):
        issues.append("缺少美食推荐（用户喜欢美食）")

    passed = len(issues) == 0
    return passed, issues


def reflection_loop(user_request: str, max_attempts: int = 3):
    """反思与自我修正模式"""

    log_divider("模式6: 反思与自我修正")
    log_step("📥", "用户输入", user_request)

    for attempt in range(1, max_attempts + 1):
        # 生成方案
        log_step("📝", f"第 {attempt} 次生成方案")
        plan = generate_plan(attempt)
        print(f"   航班: {plan['flights']}")
        print(f"   酒店: {plan['hotel']}")
        print(f"   行程: {len(plan['itinerary'])} 天")
        for day in plan['itinerary']:
            print(f"     • {day}")
        print(f"   预算: {plan['budget']}")

        # 自我评估
        log_step("🔍", f"第 {attempt} 次自我评估")
        passed, issues = evaluate_plan(plan)

        if passed:
            print("   评估结果: ✅ 方案通过所有检查")
            log_step("🎯", "最终方案确认", f"经过 {attempt} 次迭代，方案质量达标")
            return plan
        else:
            print(f"   评估结果: ❌ 发现 {len(issues)} 个问题")
            for issue in issues:
                print(f"     • {issue}")
            print(f"   → 将根据反馈修正方案...")

    log_step("⚠️", "达到最大尝试次数", "返回最新版本方案")
    return plan


reflection_loop("帮我规划5天东京旅行，喜欢文化和美食，预算中等")

### 第一次 vs 第二次

第一次方案问题很多（转机航班、胶囊旅馆、空白行程）。通过反思后 Agent 识别出所有问题，第二次方案质量大幅提升。

在实际应用中，Reviewer 可以是同一个 LLM（不同 Prompt）、不同模型、甚至一组规则检查器的组合。

---
## 模式 7：人机协作 (Human-in-the-Loop)

对于高风险决策（如付款、预订），Agent 不应完全自主——需要在关键节点 **暂停并请求人类确认**。

```
Agent 规划 → [检查点1: 确认行程] → Agent 搜索 → [检查点2: 确认预订] → Agent 执行 → 完成
                    ↑ 人类确认                        ↑ 人类确认
```

### 适用场景
- 涉及金钱的操作（预订、付款）
- 不可逆的操作（发送邮件、删除数据）
- 需要主观判断的决策（创意方向）

In [ ]:
class HumanInTheLoopAgent:
    """带人工确认检查点的 Agent"""

    def __init__(self, name: str, auto_approve: bool = True):
        self.name = name
        self.auto_approve = auto_approve
        self.checkpoints_log: list[dict] = []

    def checkpoint(self, title: str, detail: str, risk_level: str = "low") -> bool:
        """人工确认检查点"""
        icons = {"low": "🟢", "medium": "🟡", "high": "🔴"}
        icon = icons.get(risk_level, "⚪")

        log_step(f"{icon}🛑", f"检查点: {title} [风险: {risk_level}]")
        print(f"   {detail}")

        if self.auto_approve:
            approved = True
            print(f"   → 自动批准 (演示模式)")
        else:
            user_input = input(f"   → 确认执行？(y/n): ")
            approved = user_input.lower() in ('y', 'yes', '')

        self.checkpoints_log.append({
            "title": title, "risk": risk_level,
            "approved": approved
        })

        if approved:
            print(f"   ✅ 已确认，继续执行")
        else:
            print(f"   ❌ 已拒绝，流程中止")

        return approved


def human_in_the_loop(user_request: str):
    """人机协作模式"""

    log_divider("模式7: 人机协作")
    log_step("📥", "用户输入", user_request)

    agent = HumanInTheLoopAgent("TravelAgent", auto_approve=True)

    # Phase 1: 分析需求 (无需确认)
    log_step("🔍", "Phase 1: 分析需求")
    print("   目的地: 东京 | 时长: 5天 | 偏好: 文化+美食 | 预算: 中等")

    # Phase 2: 行程方案 (需要确认)
    log_step("📝", "Phase 2: 生成行程方案")
    plan = "ANA直飞¥3200 + 新宿格拉斯丽¥550/晚 + 5日文化美食行程"
    print(f"   方案: {plan}")

    if not agent.checkpoint("确认行程方案", f"即将按此方案搜索具体可预订选项:\n   {plan}", "medium"):
        return "用户取消了行程方案"

    # Phase 3: 搜索可预订选项
    log_step("🌐", "Phase 3: 搜索可预订选项")
    booking_options = {
        "flight": "ANA NH964, 4/1 09:00-13:00, ¥3,200",
        "hotel": "新宿格拉斯丽, 4/1-4/6, 含早, ¥2,750",
    }
    for k, v in booking_options.items():
        print(f"   {k}: {v}")

    # Phase 4: 预订确认 (高风险 - 涉及付款)
    total = 3200 + 2750
    if not agent.checkpoint(
        "确认预订并付款",
        f"即将预订以下项目 (总计 ¥{total:,}):\n"
        f"   ✈️ {booking_options['flight']}\n"
        f"   🏨 {booking_options['hotel']}",
        "high"
    ):
        return "用户取消了预订"

    # Phase 5: 执行预订
    log_step("⚡", "Phase 5: 执行预订")
    print("   ✈️ 航班预订成功! 确认码: ANA-2026-XK7832")
    print("   🏨 酒店预订成功! 确认码: GRA-2026-TK4521")

    # 汇总
    log_step("🎯", "任务完成")
    print(f"   共经过 {len(agent.checkpoints_log)} 个检查点:")
    for cp in agent.checkpoints_log:
        status = "✅通过" if cp['approved'] else "❌拒绝"
        print(f"     {cp['title']}: {status} (风险: {cp['risk']})")

    return "预订完成"


human_in_the_loop("帮我预订5天东京旅行，文化美食为主，预算中等")

### 风险分级

不是每个步骤都需要人工确认——那样太慢了。合理做法是 **按风险分级**：

- 🟢 **低风险**：信息查询、方案生成 → 自动执行
- 🟡 **中风险**：方案确认、偏好选择 → 可选确认
- 🔴 **高风险**：付款、预订、不可逆操作 → 必须确认

---
## 综合对比与总结

### 各模式对比

In [ ]:
comparison = [
    ["顺序链",   "⭐",    "低",  "❌", "❌", "❌", "流程固定的简单任务"],
    ["任务分解", "⭐⭐",   "中",  "✅", "❌", "❌", "可预先规划的复杂任务"],
    ["ReAct",    "⭐⭐⭐", "高",  "✅", "✅", "❌", "需要动态决策的任务"],
    ["工具调用", "⭐⭐",   "中",  "✅", "✅", "❌", "需要外部数据的任务"],
    ["多Agent",  "⭐⭐⭐", "高",  "✅", "✅", "✅", "跨领域复杂任务"],
    ["反思修正", "⭐⭐⭐", "中高", "✅", "❌", "✅", "高质量要求的任务"],
    ["人机协作", "⭐⭐",   "低",  "✅", "❌", "❌", "高风险决策任务"],
]

header = f"{'模式':<10} {'复杂度':<6} {'灵活性':<5} {'动态规划':<6} {'工具':<4} {'多Agent':<7} {'适用场景'}"
print(header)
print("─" * len(header))
for row in comparison:
    print(f"{row[0]:<10} {row[1]:<6} {row[2]:<5} {row[3]:<6} {row[4]:<4} {row[5]:<7} {row[6]}")

### 实际应用中的组合

真实的 Agent 系统通常 **组合多种模式**：

```
用户请求
  ↓
[任务分解] → 生成子任务列表
  ↓
[多Agent协作] → 各专家 Agent 认领子任务
  ↓
每个 Agent 内部用 [ReAct + 工具调用] 完成子任务
  ↓
[反思修正] → 检查整体方案质量
  ↓
[人机协作] → 关键节点请求确认
  ↓
最终输出
```

In [ ]:
def combined_agent_system(user_request: str):
    """组合模式: 展示真实系统如何融合多种模式"""

    log_divider("组合模式: 完整Agent系统")
    log_step("📥", "用户输入", user_request)

    # 1. 任务分解
    log_step("🧠", "[任务分解] Planner 分析任务")
    subtasks = ["搜索航班", "搜索酒店", "规划行程", "计算预算"]
    print(f"   子任务: {' → '.join(subtasks)}")

    # 2. 多Agent分工 + 工具调用
    log_step("👥", "[多Agent + 工具调用] 专家执行")
    results = {}
    expert_work = [
        ("✈️ FlightExpert", "search_flights()", "ANA直飞¥3200"),
        ("🏨 HotelExpert",  "search_hotels()",  "格拉斯丽¥550/晚"),
        ("🗺️ ItineraryExpert", "get_attractions()", "5日文化美食行程"),
        ("💰 BudgetExpert", "calculate_budget()", "总计¥9250"),
    ]
    for expert, tool, result in expert_work:
        print(f"   {expert} 调用 {tool} → {result}")
        results[expert.split()[1]] = result

    # 3. ReAct: 发现问题并调整
    log_step("💭", "[ReAct] 发现 Day4 行程与天气冲突")
    print("   Thought: Day4 预报有雨，teamLab 是室内项目不受影响，但台场海滨不适合")
    print("   Action: 将台场替换为室内的数字艺术博物馆")
    print("   Observation: 行程调整完成")

    # 4. 反思修正
    log_step("🔍", "[反思] Reviewer 检查方案")
    print("   ✅ 航班: 直飞，时间合理")
    print("   ✅ 酒店: 位置好，含早餐")
    print("   ✅ 行程: 每天充实，包含美食")
    print("   ✅ 预算: 在中等范围内")
    print("   → 方案质量评分: 9.2/10")

    # 5. 人机确认
    log_step("🛑", "[人机协作] 请求用户确认")
    print("   方案概要: 5天东京之旅 | ANA直飞 | 新宿格拉斯丽 | ¥9250")
    print("   → 用户确认: ✅")

    log_step("🎉", "最终方案输出",
             "5天东京文化美食之旅\n"
             "航班: ANA NH964 直飞 ¥3200\n"
             "酒店: 新宿格拉斯丽 ¥550/晚\n"
             "行程: 浅草→秋叶原→明治神宫→teamLab→筑地\n"
             "预算: ¥9250 + ¥1000应急 = ¥10250")

    print(f"\n  📊 使用了 5 种模式: 任务分解 + 多Agent + 工具调用 + ReAct + 反思 + 人机协作")


combined_agent_system("帮我规划5天东京旅行，喜欢文化和美食，预算中等")

---
## 下一步

本 Notebook 用模拟函数展示了 7 种 Agent 长任务模式的核心逻辑。要在真实项目中应用：

1. **将模拟函数替换为 LLM 调用** — 参考本课程各 Lesson 中的 Semantic Kernel / AutoGen 示例
2. **Lesson 03** → 基础 Agent + 工具注册
3. **Lesson 04** → 工具调用 (Tool Use) 深入
4. **Lesson 07** → 任务分解 (Planning) 的完整实现
5. **Lesson 08** → 多 Agent 协作的详细代码
6. **Lesson 09** → 反思与元认知 (Metacognition)
7. **Lesson 11** → MCP (Model Context Protocol) 集成外部工具

### 推荐阅读

- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761)
- [AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation](https://arxiv.org/abs/2308.08155)
- [Reflexion: Language Agents with Verbal Reinforcement Learning](https://arxiv.org/abs/2303.11366)